In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import cv2
import pandas as pd
import zipfile
import os
import time
import gc
from PIL import Image
from tqdm.notebook import tqdm
from utils import MetricsEngine

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [2]:
def load_and_prep_data(image_path, downscale_factor=4):
    """Loads image, generates raw coords, and locks everything to GPU."""
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    new_w, new_h = w // downscale_factor, h // downscale_factor
    img = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    
    img_np = np.array(img)
    img_norm = img_np / 255.0
    
    y_coords = np.linspace(-1, 1, new_h)
    x_coords = np.linspace(-1, 1, new_w)
    grid_x, grid_y = np.meshgrid(x_coords, y_coords)
    coords_raw = torch.tensor(np.stack([grid_x.flatten(), grid_y.flatten()], axis=-1), dtype=torch.float32)
    colors = torch.tensor(img_norm.reshape(-1, 3), dtype=torch.float32)
    
    return {
        "coords_raw": coords_raw.to(device),
        "colors": colors.to(device),
        "h": new_h, "w": new_w,
        "num_pixels": new_h * new_w,
        "original_np": img_np
    }

dataset = load_and_prep_data("jcsmr-1.jpg", downscale_factor=4)

In [3]:
class VectorizedHashGrid2D(nn.Module):
    def __init__(self, num_levels=12, base_res=16, max_res=1024, feature_dim=2, log2_hashmap_size=13):
        super().__init__()
        self.num_levels = num_levels
        self.feature_dim = feature_dim
        self.max_entries = 2 ** log2_hashmap_size
        self.hash_mask = self.max_entries - 1  # Used for ultra-fast bitwise AND instead of modulo

        growth_factor = np.exp((np.log(max_res) - np.log(base_res)) / (num_levels - 1))
        resolutions = [int(base_res * (growth_factor ** i)) for i in range(num_levels)]

        table_sizes = []
        is_dense = []
        for res in resolutions:
            tsize = min(res * res, self.max_entries)
            table_sizes.append(tsize)
            is_dense.append(res * res <= self.max_entries)

        self.num_dense = sum(is_dense)
        self.num_hash = num_levels - self.num_dense

        # Create single flat parameter array for F.embedding
        self.total_params = sum(table_sizes)
        self.embeddings = nn.Parameter(torch.empty(self.total_params, feature_dim))
        nn.init.uniform_(self.embeddings, a=-1e-4, b=1e-4)

        offsets = [0]
        for i in range(len(table_sizes) - 1):
            offsets.append(offsets[-1] + table_sizes[i])

        # Buffers cast to int32 for bandwidth efficiency
        self.register_buffer("res_int", torch.tensor(resolutions, dtype=torch.int32).view(1, -1))
        self.register_buffer("offsets", torch.tensor(offsets, dtype=torch.int32).view(1, -1))
        
        # Prime numbers for hashing
        self.p0 = 1
        self.p1 = 2654435761

        self.decoder = nn.Sequential(
            nn.Linear(num_levels * feature_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )

    def forward(self, coords):
        # Map [-1, 1] to [0, 1]
        coords = (coords + 1) * 0.5  

        # Broadcast and scale coordinates
        x = coords[:, 0:1] * (self.res_int - 1).float()
        y = coords[:, 1:2] * (self.res_int - 1).float()

        # FIX #7: Direct int32 cast (equivalent to floor for positive numbers, but faster)
        x0 = x.to(torch.int32)
        y0 = y.to(torch.int32)
        
        x1 = torch.clamp(x0 + 1, max=self.res_int - 1)
        y1 = torch.clamp(y0 + 1, max=self.res_int - 1)

        # Interpolation weights [B, L, 1]
        wx = (x - x0).unsqueeze(-1)
        wy = (y - y0).unsqueeze(-1)

        h00_list, h10_list, h01_list, h11_list = [], [], [], []

        # --- DENSE LEVELS ---
        if self.num_dense > 0:
            x0_d, y0_d = x0[:, :self.num_dense], y0[:, :self.num_dense]
            x1_d, y1_d = x1[:, :self.num_dense], y1[:, :self.num_dense]
            res_d = self.res_int[:, :self.num_dense]
            
            h00_list.append(y0_d * res_d + x0_d)
            h10_list.append(y0_d * res_d + x1_d)
            h01_list.append(y1_d * res_d + x0_d)
            h11_list.append(y1_d * res_d + x1_d)

        # --- HASH LEVELS (FIX #2: Using bitwise XOR ^ and bitwise AND & mask) ---
        if self.num_hash > 0:
            x0_h, y0_h = x0[:, self.num_dense:], y0[:, self.num_dense:]
            x1_h, y1_h = x1[:, self.num_dense:], y1[:, self.num_dense:]
            
            h00_list.append(((x0_h * self.p0) ^ (y0_h * self.p1)) & self.hash_mask)
            h10_list.append(((x1_h * self.p0) ^ (y0_h * self.p1)) & self.hash_mask)
            h01_list.append(((x0_h * self.p0) ^ (y1_h * self.p1)) & self.hash_mask)
            h11_list.append(((x1_h * self.p0) ^ (y1_h * self.p1)) & self.hash_mask)

        # Combine and add global offsets
        h00 = torch.cat(h00_list, dim=1) + self.offsets
        h10 = torch.cat(h10_list, dim=1) + self.offsets
        h01 = torch.cat(h01_list, dim=1) + self.offsets
        h11 = torch.cat(h11_list, dim=1) + self.offsets

        # FIX #1: Use F.embedding instead of manual array indexing
        f00 = F.embedding(h00, self.embeddings)
        f10 = F.embedding(h10, self.embeddings)
        f01 = F.embedding(h01, self.embeddings)
        f11 = F.embedding(h11, self.embeddings)

        # FIX #8: Single-step fused polynomial interpolation
        w00 = (1 - wx) * (1 - wy)
        w10 = wx * (1 - wy)
        w01 = (1 - wx) * wy
        w11 = wx * wy
        f = f00 * w00 + f10 * w10 + f01 * w01 + f11 * w11

        f = f.view(coords.shape[0], self.num_levels * self.feature_dim)
        return torch.sigmoid(self.decoder(f))

In [4]:
def train_and_evaluate(config, dataset):
    model_name = config["name"]
    print(f"\n=== Starting: {model_name} ===")
    
    model = VectorizedHashGrid2D(
        num_levels=config["levels"],
        base_res=16,
        max_res=config["max_res"],
        feature_dim=config["feat_dim"],
        log2_hashmap_size=config["log2"]
    ).to(device)
    
    input_data = dataset["coords_raw"]
    optimizer = optim.Adam(model.parameters(), lr=config["lr"], eps=1e-15)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config["epochs"], eta_min=1e-5)
    criterion = nn.MSELoss()
    scaler = torch.amp.GradScaler('cuda')
    
    batch_size = config["batch_size"]
    num_pixels = dataset["num_pixels"]
    
    model.train()
    pbar = tqdm(range(config["epochs"]), desc=model_name)
    
    for epoch in pbar:
        indices = torch.randperm(num_pixels, device=device)
        epoch_loss = 0.0
        batches = 0
        
        for i in range(0, num_pixels, batch_size):
            batch_idx = indices[i : i + batch_size]
            b_coords = input_data[batch_idx]
            b_colors = dataset["colors"][batch_idx]
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                pred = model(b_coords)
                loss = criterion(pred, b_colors)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            # FIX #3: Do NOT use .item() inside the inner loop (Removes CUDA sync block)
            epoch_loss += loss.detach()
            batches += 1
            
        scheduler.step()
        
        # Only pull to CPU/UI every 50 epochs
        if epoch % 2 == 0:
            avg_loss_cpu = (epoch_loss / batches).float().cpu().item()
            psnr = 10 * np.log10(1.0 / avg_loss_cpu)
            pbar.set_postfix({"PSNR": f"{psnr:.2f}", "LR": f"{scheduler.get_last_lr()[0]:.1e}"})

    model.eval()
    start_time = time.time()
    predicted_colors = []
    
    with torch.no_grad(), torch.amp.autocast('cuda'):
        chunk_size = 65536 
        for i in range(0, num_pixels, chunk_size):
            chunk = input_data[i : i + chunk_size]
            predicted_colors.append(model(chunk))
            
        full_pred = torch.cat(predicted_colors, dim=0)
        
    torch.cuda.synchronize()
    latency_ms = (time.time() - start_time) * 1000

    pred_np = full_pred.float().cpu().numpy().reshape(dataset["h"], dataset["w"], 3)
    pred_img_uint8 = (pred_np * 255).clip(0, 255).astype(np.uint8)
    os.makedirs("images", exist_ok=True)
    cv2.imwrite(f"images/{model_name}.png", cv2.cvtColor(pred_img_uint8, cv2.COLOR_RGB2BGR))

    # FIX #6: Safe Saving. Process to half-precision strictly on CPU to avoid VRAM spikes.
    os.makedirs("models", exist_ok=True)
    pth_path = f"models/{model_name}.pth"
    zip_path = f"models/{model_name}.zip"
    
    state_dict_cpu_half = {k: v.cpu().half() for k, v in model.state_dict().items()}
    torch.save(state_dict_cpu_half, pth_path)
    
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_LZMA) as zipf:
        zipf.write(pth_path, arcname=f"{model_name}.pth")
    size_kb = os.path.getsize(zip_path) / 1024.0

    # Compute Metrics
    engine = MetricsEngine(device)
    metrics = engine.compute_all(dataset["original_np"], pred_img_uint8)

    row = {"Method": model_name, "Size_KB": round(size_kb, 2), "Latency_ms": round(latency_ms, 2)}
    row.update(metrics)

    del model, optimizer, scheduler, scaler, full_pred, predicted_colors, criterion, state_dict_cpu_half
    gc.collect()
    torch.cuda.empty_cache()
    
    return row

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 12,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 1e-2
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,427.14,40.95,35.44208323122413,0.9253563858126324,0.997327529249738,0.28126755356788635

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 12,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 5e-3
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,427.91,41.47,35.001696150731235,0.9208701678428887,0.9970406879283622,0.2822205424308777

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 8,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 5e-3
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,284.25,30.96,33.09079182810433,0.8966360558143583,0.9953868989399589,0.3403562307357788

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 8,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 1e-2
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,286.0,29.48,33.26206804454861,0.8998128146188288,0.9955651120801258,0.33177825808525085

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 8,       
        "max_res": 1024,    
        "feat_dim": 3, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 1e-2
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,423.41,32.64,34.97335481651752,0.9216594076743599,0.9970295619346243,0.2822382152080536

best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 7,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": best_batchsize, 
        "lr": 1e-2
    }
]

Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
Hash_Q30_Vector,212.58,51.94,31.82696034122623,0.8751278639034478,0.9938130938545062,0.3677709996700287


In [ ]:
# FIX #5: Set Batch Size to 16k. 
# Full-image batching kills the optimizer. 16,384 allows fast cache hits and frequent optimizer updates.
best_batchsize = 16384

EXPERIMENTS = [
    {
        "name": "Hash_Q30_Vector", 
        "levels": 12,       
        "max_res": 1024,    
        "feat_dim": 2, 
        "log2": 14,         
        "epochs": 300,
        "batch_size": int(best_batchsize), 
        "lr": 1e-2
    }
]

os.makedirs("results", exist_ok=True) 
all_neural_results = []

for config in EXPERIMENTS: 
    result = train_and_evaluate(config, dataset)
    all_neural_results.append(result)
    
df_neural = pd.DataFrame(all_neural_results)
df_neural.to_csv("results/neural_metrics.csv", index=False) 
display(df_neural)


=== Starting: Hash_Q30_Vector ===


Hash_Q30_Vector:   0%|          | 0/300 [00:00<?, ?it/s]

,Method,Size_KB,Latency_ms,PSNR,SSIM,NCE,LPIPS
0,Hash_Q30_Vector,429.55,40.57,35.882747,0.930545,0.997601,0.26375
